# eg10 (v2 replay) — slider `k` shows layers 1..k, slider `n_1` shows the first n_1 objects of layer k

v1 (`examples/eg10_slider.ipynb`, main): `ggb.listen` + `shared_objects` + `on_shared_update` callbacks → `setLayerVisible` / `setVisible` (Apps API) and a `getXML`/`evalXML` round trip to reset `n_1`.

v2: the same file (`2025_06_08.ggb`, its own sliders `k`, `n_1`) loaded by `xml_in`; layers from the XML DataFrame (C6); the two reactions are `ShowLayer`/`HideLayer` and `SetVisibleInView` scripting commands sent by `eval` (no new verb); `n_1` is reset by `SetValue`; the cell waits with a blocking pull (`events(wait=…)`), one reaction per user operation.

In [1]:
import sys, time; sys.path.insert(0, '/Users/manabu/work/ggblab-replay')
import polars as pl
from ggblab_extra import ConstructionIO, read_ggb
import ggblab.host.html_host as H; H.DEPLOY = 'https://cdn.geogebra.org/apps/deployggb.js'
from ggblab import GeoGebra

In [2]:
g = GeoGebra(appName='suite', showToolBar=True, showAlgebraInput=True); g

## load the v1 file by `xml_in`, read the layers back from the XML (C6 data-in)

In [3]:
xml0 = read_ggb('2025_06_08.ggb'); t0 = time.time(); r = g.set_xml(xml0, timeout=60); xml = g.xml(timeout=60)
df = ConstructionIO.from_xml(xml); print('ROWS', df.height, 'in', round(time.time() - t0, 2), 's; sliders k =', g.value('k'), 'n_1 =', g.value('n_1'))
by_layer = {int(k): v for k, v in df.group_by('Layer').agg(pl.col('Name')).sort('Layer').iter_rows()}
print({k: len(v) for k, v in by_layer.items()})

In [4]:
def visible_per_layer():
    d = ConstructionIO.from_xml(g.xml(timeout=60))
    return {int(k): int(v) for k, v in d.group_by('Layer').agg(pl.col('ShowObject').sum()).sort('Layer').iter_rows()}

def wait_any(labels, timeout=30.0):
    """block until an update of one of `labels` arrives (pull); returns the event or None"""
    t0 = time.time()
    while time.time() - t0 < timeout:
        for e in g.events(wait=min(timeout - (time.time() - t0), g.SLICE)):
            if e.get('type') in ('update', 'add') and e.get('label') in labels: return e
    return None

r = g.command(*[f'HideLayer({k})' for k in range(1, 10)], 'SetValue(k, 0)', 'SetValue(n_1, 0)', timeout=30)
print('start ->', visible_per_layer()); _ = g.events()   # drain the backlog before the loop

## the loop: `k` → layers, then `n_1` → objects within layer k (v1 eg10 semantics)

The browser-side driver moves the sliders (`SetValue(k, 3)`, then `SetValue(n_1, 2)`, `SetValue(n_1, 5)`, then `SetValue(k, 5)`); the cell reacts once per operation and prints what it did.

In [7]:
cur_k, steps, log, t_all = 0, 0, [], time.time()
while steps < 4 and time.time() - t_all < 150:
    t0 = time.time(); e = wait_any(('k', 'n_1'), timeout=40)
    if e is None: print('  no change within 40 s'); break
    dt = round(time.time() - t0, 2)
    if e['label'] == 'k':
        n = int(round(float(g.value('k'))))
        if n == cur_k: continue                      # the same value again (one SetValue can fire several update events)
        r = g.command(*[(f'ShowLayer({j})' if j <= n else f'HideLayer({j})') for j in range(1, 10)], 'SetValue(n_1, 0)', timeout=30)
        cur_k = n; _ = g.events()                    # our own SetValue(n_1, 0) fires update:n_1 — drain it (one reaction per user operation)
        vis = visible_per_layer(); steps += 1; log.append(('k', n, dt))
        print(f'step {steps}: k -> {n} (woke after {dt} s): layers 1..{n} shown, n_1 reset; visible per layer {vis}')
    else:
        m = int(round(float(g.value('n_1')))); names = by_layer.get(cur_k, [])
        r = g.command(*[f'SetVisibleInView({nm}, 1, {"true" if i < m else "false"})' for i, nm in enumerate(names)], timeout=30)
        vis = visible_per_layer(); steps += 1; log.append(('n_1', m, dt))
        print(f'step {steps}: n_1 -> {m} (woke after {dt} s): first {m} of {len(names)} objects of layer {cur_k} shown; visible per layer {vis}')
print('LOG', log)

In [6]:
print('DONE', 'errors:', g.errors())